# TP 3 — Partie 2 : du notebook au pipeline automatisé

**Objectif :** que le traitement des photos tourne **seul**, chaque jour, sans que personne n'ouvre de notebook. Il faut pour cela :
1. transformer le code du notebook en **scripts**, un par étape, que l'on peut lancer sans JupyterLab ;
2. décrire l'enchaînement des étapes dans un **DAG** (*Directed Acyclic Graph*) Apache Airflow : quelle étape attend quelle autre, quand le pipeline démarre, que faire en cas d'erreur.

| Étape (script) | Rôle |
|---|---|
| `bronze_photos.py` | ingère les **nouvelles** photos de la zone d'atterrissage, et les fiches produits |
| `silver_photos.py` | décode et contrôle les nouvelles photos ; quarantaine |
| `gold_photos.py` | vecteurs des nouvelles photos ; contrôle des types de produits |
| `controle_qualite.py` | arrête le pipeline si trop de photos sont rejetées |

> **⚠️ Mémoire.** Airflow exécute les scripts dans son propre conteneur, avec sa propre session Spark. Arrêtez le noyau de ce notebook (*Kernel > Shut Down Kernel*) avant de lancer le DAG, et Trino/Superset s'ils tournent.

## Étape 1 — Repartir de zéro

Le pipeline doit tout reconstruire lui-même. On supprime donc les tables créées dans le notebook 01, et le fichier qui mémorise les photos déjà ingérées.

In [ ]:
import os
import boto3
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("tp3-reset").getOrCreate()
for t in ["bronze.photos", "bronze.produits_brut", "silver.photos", "silver.quarantaine_photos",
          "gold.photos_vecteurs", "gold.photos_controle"]:
    spark.sql(f"DROP TABLE IF EXISTS lakehouse.{t} PURGE")
boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"]).delete_object(Bucket="landing", Key="catalogue/_ingerees.txt")
spark.stop()
print("tables du TP 3 supprimées ; les photos du lot 1 sont toujours dans la zone d'atterrissage")

## Étape 2 — Les scripts

Ouvrez les fichiers du dossier `pipeline/`. `commun.py` regroupe les fonctions partagées ; `bronze_photos.py` est complet : lisez-le. Il est **incrémental** : il compare les photos de la zone d'atterrissage à la liste des photos déjà ingérées, et n'ajoute que les nouvelles.

Complétez `silver_photos.py` (règles de qualité, à reprendre du notebook 01) et `controle_qualite.py` (seuil de rejet), en suivant les commentaires `# À COMPLÉTER`. Testez chaque script **à la main**, dans l'ordre, avant de confier le tout à Airflow :

In [ ]:
!cd pipeline && python bronze_photos.py

In [ ]:
!cd pipeline && python silver_photos.py

In [ ]:
!cd pipeline && python gold_photos.py

In [ ]:
!cd pipeline && python controle_qualite.py

Relancez `bronze_photos.py` : il ne doit **rien** ajouter.

In [ ]:
!cd pipeline && python bronze_photos.py

> **❓ Question 1.** Un script est **idempotent** quand on peut le relancer sans changer le résultat. Pourquoi est-ce indispensable dans un pipeline automatisé ? Que se passerait-il si `bronze_photos.py` rechargeait tout le dossier à chaque exécution ?

## Étape 3 — Le DAG Airflow

Copiez le squelette du DAG dans le dossier que lit Airflow, puis complétez-le :
- un capteur (`PythonSensor`) qui attend de nouvelles photos dans la zone d'atterrissage ;
- une tâche `BashOperator` par script, puis `dbt build` sur le projet du TP 2 ;
- l'ordre des tâches, avec l'opérateur `>>`.

In [ ]:
!mkdir -p dags && cp -n dag_catalogue_photos.py dags/catalogue_photos.py && ls dags

Démarrez Airflow depuis le terminal **Ubuntu** de votre ordinateur (pas celui de JupyterLab), dans le dossier du dépôt :
```bash
docker compose --profile orchestration up -d
```
Ouvrez http://localhost:8085. Le DAG `catalogue_photos` apparaît en moins d'une minute (sinon, l'onglet *Import errors* indique l'erreur dans votre fichier).

1. Activez le DAG (interrupteur), puis lancez-le (*Trigger*).
2. Suivez l'exécution dans la vue **Graph**, puis ouvrez les **journaux** (*Logs*) de chaque tâche.
3. Dans la vue **Gantt**, relevez la durée de chaque tâche.

## Étape 4 — Le lot 2 arrive

Les vendeurs 09 à 12 envoient leurs photos. Déposez-les, relancez le DAG, et comparez la durée de chaque tâche à celle de la première exécution.

In [ ]:
!python /opt/lakehouse/scripts/deposer_photos.py --lot 2

> **❓ Question 2.** Lors de la seconde exécution, combien de photos chaque étape a-t-elle traitées ? Pourquoi `gold_photos.py` réentraîne-t-il quand même son classifieur sur toutes les photos ?

## Étape 5 — Quand une étape échoue

Dans `controle_qualite.py`, abaissez temporairement le seuil de rejet à 1 %, enregistrez, puis relancez le DAG.
1. Que devient la tâche `controle_qualite` ? Et `dbt_build` ?
2. Combien de fois Airflow a-t-il tenté la tâche avant d'abandonner ? Où ce nombre est-il réglé ?
3. Remettez le seuil à 10 % et relancez **seulement** la tâche en échec (*Clear*).

> **❓ Question 3.** Pourquoi est-il préférable d'arrêter tout le pipeline plutôt que de laisser passer un lot douteux ? À quoi servent les relances automatiques (`retries`), et pour quel type d'erreur sont-elles inutiles ?

> **❓ Question 4.** Le DAG est planifié `@daily` avec `catchup=False`. Que signifient ces deux réglages ? Qu'aurait fait Airflow avec `catchup=True` et `start_date` au 1er janvier 2026 ?